In [1]:
from pathlib import Path
from torch.utils.data import DataLoader
from rf_learning_dataset import RFLearningDataset

EXP = {
    "experiment_name": "tiny_random64_pilot80",
    "project_root": "/home/liujia/RF_Image",

    "include_categories": ["carotid", "muscle", "phantom"],

    "batch_size": 4,
    "num_epochs": 100,
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "normalize": True,
    "abs_weight": 0.1,

    "model_name": "tiny",
    "hidden": 64,
    "seed": 20260522,
}

PROJECT_ROOT = Path(EXP["project_root"])
DATA_ROOT = PROJECT_ROOT / "RF_LearningSamples_random64_pilot80"

CKPT_DIR = PROJECT_ROOT / "checkpoint" / EXP["experiment_name"]
METRIC_DIR = PROJECT_ROOT / "test_metrics" / EXP["experiment_name"]
VIS_DIR = PROJECT_ROOT / "vis_best_model" / EXP["experiment_name"]

CKPT_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)
VIS_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT :", DATA_ROOT)
print("CKPT_DIR  :", CKPT_DIR)
print("METRIC_DIR:", METRIC_DIR)
print("VIS_DIR   :", VIS_DIR)

train_set = RFLearningDataset(
    root_dir=DATA_ROOT / "train",
    sample_group="/sample_000001",
    normalize=EXP["normalize"],
    include_categories=EXP["include_categories"],
)

val_set = RFLearningDataset(
    root_dir=DATA_ROOT / "val",
    sample_group="/sample_000001",
    normalize=EXP["normalize"],
    include_categories=EXP["include_categories"],
)

test_set = RFLearningDataset(
    root_dir=DATA_ROOT / "test",
    sample_group="/sample_000001",
    normalize=EXP["normalize"],
    include_categories=EXP["include_categories"],
)

expected_categories = set(EXP["include_categories"])
for split_name, dataset in [
    ("train", train_set),
    ("val", val_set),
    ("test", test_set),
]:
    actual_categories = set(dataset.categories)
    assert actual_categories <= expected_categories, (
        f"Unexpected category in {split_name}: {sorted(actual_categories)}"
    )

loader_kwargs = {
    "batch_size": EXP["batch_size"],
    "num_workers": 0,
    "pin_memory": True,
}

train_loader = DataLoader(
    train_set,
    shuffle=True,
    **loader_kwargs,
)

val_loader = DataLoader(
    val_set,
    shuffle=False,
    **loader_kwargs,
)

test_loader = DataLoader(
    test_set,
    shuffle=False,
    **loader_kwargs,
)

print("Train:", len(train_set))
print("Val  :", len(val_set))
print("Test :", len(test_set))

batch = next(iter(train_loader))
print("input   :", batch["input"].shape)
print("label   :", batch["label"].shape)
print("baseline:", batch["baseline"].shape)
print("category:", batch["category"])


DATA_ROOT : /home/liujia/RF_Image/RF_LearningSamples_random64_pilot80
CKPT_DIR  : /home/liujia/RF_Image/checkpoint/tiny_random64_pilot80
METRIC_DIR: /home/liujia/RF_Image/test_metrics/tiny_random64_pilot80
VIS_DIR   : /home/liujia/RF_Image/vis_best_model/tiny_random64_pilot80
RFLearningDataset
  root_dir   : /home/liujia/RF_Image/RF_LearningSamples_random64_pilot80/train
  samples    : 57
  normalize  : True
  include    : {'muscle', 'carotid', 'phantom'}
  exclude    : set()
  carotid   : 19
  muscle    : 19
  phantom   : 19
RFLearningDataset
  root_dir   : /home/liujia/RF_Image/RF_LearningSamples_random64_pilot80/val
  samples    : 12
  normalize  : True
  include    : {'muscle', 'carotid', 'phantom'}
  exclude    : set()
  carotid   : 4
  muscle    : 4
  phantom   : 4
RFLearningDataset
  root_dir   : /home/liujia/RF_Image/RF_LearningSamples_random64_pilot80/test
  samples    : 12
  normalize  : True
  include    : {'muscle', 'carotid', 'phantom'}
  exclude    : set()
  carotid   : 4

In [2]:
from rf_models import build_model
from rf_train_utils import (
    seed_everything,
    get_device,
    count_trainable_parameters,
    train_model_jupyter,
    plot_training_curve,
)

seed_everything(EXP["seed"])

device = get_device()
print("Device:", device)

model = build_model(
    EXP["model_name"],
    in_channels=1536,
    hidden=EXP["hidden"],
    out_channels=2,
).to(device)

print(f"Trainable parameters: {count_trainable_parameters(model) / 1e6:.3f} M")

history = train_model_jupyter(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    ckpt_dir=CKPT_DIR,
    experiment_name=EXP["experiment_name"],
    num_epochs=EXP["num_epochs"],
    lr=EXP["lr"],
    weight_decay=EXP["weight_decay"],
    abs_weight=EXP["abs_weight"],
    print_every=5,
    seed=EXP["seed"],
    config=EXP,
    patience=15,
)


Device: cuda
Trainable parameters: 0.265 M

RF TRAINING STARTUP CHECK
experiment_name      : tiny_random64_pilot80
ckpt_dir             : /home/liujia/RF_Image/checkpoint/tiny_random64_pilot80
model_class          : TinyResidualRFNet
trainable_parameters : 264,738

Initial validation:
val_L1=2.830866e-01 | baseline_L1=2.830866e-01 | improvement=0.00%
Epoch 0001 | train_L1=2.643025e-01 | train_base=3.088882e-01 | train_impr= 14.43% | val_L1=1.859537e-01 | val_base=2.830866e-01 | val_impr= 34.31% | lr=1.00e-03
Epoch 0005 | train_L1=1.601416e-01 | train_base=3.056163e-01 | train_impr= 47.60% | val_L1=1.505987e-01 | val_base=2.830866e-01 | val_impr= 46.80% | lr=9.94e-04
Epoch 0010 | train_L1=1.620059e-01 | train_base=3.223826e-01 | train_impr= 49.75% | val_L1=1.494356e-01 | val_base=2.830866e-01 | val_impr= 47.21% | lr=9.76e-04
Epoch 0015 | train_L1=1.513689e-01 | train_base=3.111408e-01 | train_impr= 51.35% | val_L1=1.553557e-01 | val_base=2.830866e-01 | val_impr= 45.12% | lr=9.46e-04
Epo

In [3]:
import torch
import pandas as pd

from rf_models import build_model
from rf_eval_utils import (
    evaluate_full_test_set,
    summarize_test_metrics,
    save_test_summaries,
    find_worse_samples,
)

from rf_visualization import (
    visualize_model_samples,
    select_indices_from_metrics_df,
)

# ============================================================
# Load best model
# ============================================================

best_model = build_model(
    EXP["model_name"],
    in_channels=1536,
    hidden=EXP["hidden"],
    out_channels=2,
).to(device)

ckpt_path = CKPT_DIR / "best_model.pth"
ckpt = torch.load(ckpt_path, map_location=device)

best_model.load_state_dict(ckpt["model"])
best_model.eval()

print("Loaded best model")
print("  epoch       :", ckpt["epoch"])
print("  best val L1 :", ckpt["best_val_l1"])
print("  ckpt path   :", ckpt_path)

# ============================================================
# Full test evaluation
# ============================================================

df_test = evaluate_full_test_set(
    model=best_model,
    dataset=test_set,
    device=device,
    batch_size=EXP["batch_size"],
    save_csv_path=METRIC_DIR / "test_per_sample_metrics.csv",
)

overall, cat_summary = summarize_test_metrics(df_test)

save_test_summaries(
    df=df_test,
    metric_dir=METRIC_DIR,
    overall=overall,
    cat_summary=cat_summary,
    prefix="test",
)

# ============================================================
# Worse samples
# ============================================================

worse_complex = find_worse_samples(
    df_test,
    metric="complex",
    top_k=20,
)

worse_abs = find_worse_samples(
    df_test,
    metric="abs",
    top_k=20,
)

worse_complex.to_csv(METRIC_DIR / "test_worse_complex_top20.csv", index=False)
worse_abs.to_csv(METRIC_DIR / "test_worse_abs_top20.csv", index=False)

# ============================================================
# Visualization: random/category samples
# ============================================================

vis_random = visualize_model_samples(
    model=best_model,
    dataset=test_set,
    device=device,
    save_dir=VIS_DIR / "random_by_category",
    indices=None,
    n_per_category=3,
    categories=EXP["include_categories"],
    view="xz",
    slice_index=None,
    db_min=-60,
    show=False,
    prefix="random",
)

# ============================================================
# Visualization: worst complex samples
# ============================================================

worst_indices = select_indices_from_metrics_df(
    dataset=test_set,
    df=df_test,
    top_k=9,
    metric="complex_improvement",
    ascending=True,
)

vis_worst = visualize_model_samples(
    model=best_model,
    dataset=test_set,
    device=device,
    save_dir=VIS_DIR / "worst_complex",
    indices=worst_indices,
    view="xz",
    slice_index=None,
    db_min=-60,
    show=False,
    prefix="worst_complex",
)

print("\nEvaluation package finished.")
print("Metric dir:", METRIC_DIR)
print("Vis dir   :", VIS_DIR)
print("Random visualizations:", len(vis_random))
print("Worst visualizations :", len(vis_worst))


Loaded best model
  epoch       : 10
  best val L1 : 0.1494355946779251
  ckpt path   : /home/liujia/RF_Image/checkpoint/tiny_random64_pilot80/best_model.pth
Saved per-sample metrics to: /home/liujia/RF_Image/test_metrics/tiny_random64_pilot80/test_per_sample_metrics.csv

================ Overall test summary ================
Samples: 12

[Complex L1]
pred mean       : 2.5027e+03
baseline mean   : 5.1637e+03
mean improvement: 51.39%
better rate     : 100.00%

[Envelope abs L1]
pred mean       : 2.2009e+03
baseline mean   : 5.8326e+03
mean improvement: 62.73%
better rate     : 100.00%

================ Per-category summary ================


,n,complex_pred_mean,complex_base_mean,complex_improvement_mean,complex_better_rate,abs_pred_mean,abs_base_mean,abs_improvement_mean,abs_better_rate
category,,,,,,,,,
carotid,4,4193.099701,8854.497009,0.536101,1.0,3756.949020,10452.776154,0.663250,1.0
muscle,4,2489.464645,4996.522400,0.510145,1.0,2200.962387,5334.750122,0.594771,1.0
phantom,4,825.586563,1640.227020,0.495424,1.0,644.695671,1710.401199,0.623866,1.0


Saved test summaries to: /home/liujia/RF_Image/test_metrics/tiny_random64_pilot80
complex worse samples: 0
abs worse samples: 0
Selected indices: [0, 1, 2, 4, 5, 6, 8, 9, 10]
Saved: /home/liujia/RF_Image/vis_best_model/tiny_random64_pilot80/random_by_category/001_random_carotid_RF000486_carotid_test_patch001.png
  complex L1 pred/base: 5.2733e+03 / 1.2307e+04
  abs     L1 pred/base: 4.5093e+03 / 1.4354e+04
Saved: /home/liujia/RF_Image/vis_best_model/tiny_random64_pilot80/random_by_category/002_random_carotid_RF000487_carotid_test_patch001.png
  complex L1 pred/base: 5.5172e+02 / 1.1063e+03
  abs     L1 pred/base: 3.9853e+02 / 1.1876e+03
Saved: /home/liujia/RF_Image/vis_best_model/tiny_random64_pilot80/random_by_category/003_random_carotid_RF000488_carotid_test_patch001.png
  complex L1 pred/base: 2.6737e+03 / 7.3446e+03
  abs     L1 pred/base: 2.0739e+03 / 8.3753e+03
Saved: /home/liujia/RF_Image/vis_best_model/tiny_random64_pilot80/random_by_category/004_random_muscle_RF000386_muscle_t